# 🚀 Skip Flow Test

**Goal:** Test the "Skip, search now" flow where user bypasses clarification questions.

**Flow:**
```
User: "laptop"
    ↓
[Skip clarification]
    ↓
Search with original keyword: "laptop"
    ↓
Full Pipeline → Results
```

**Use Case:** User doesn't want to answer questions, wants quick results.

In [1]:
# Cell 1: Setup & Imports
import os
import sys
import logging

# Change working directory to segment4
os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

print(f"Working directory: {os.getcwd()}")

# Setup logging
logging.basicConfig(level=logging.INFO)
root = logging.getLogger()
root.setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

print("✅ Setup complete!")

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
✅ Setup complete!


In [2]:
# Cell 2: Import Pipeline Components
from typing import List

# Import từ bestbuy modules
from price_agents.bestbuy_deals import (
    ScrapedBestBuyDeal,
    filter_sale_urls,
    scrape_bestbuy_products
)
from price_agents.bestbuy_scanner_agent import (
    BestBuySearchAgent,
    BestBuyScannerAgent
)
from price_agents.deals import Deal, DealSelection, Opportunity

print("✅ Pipeline components imported!")
print("   - BestBuySearchAgent (Brave MCP)")
print("   - filter_sale_urls")
print("   - scrape_bestbuy_products (Playwright)")
print("   - BestBuyScannerAgent (GPT-5-mini)")

✅ Pipeline components imported!
   - BestBuySearchAgent (Brave MCP)
   - filter_sale_urls
   - scrape_bestbuy_products (Playwright)
   - BestBuyScannerAgent (GPT-5-mini)


In [3]:
# Cell 3: Initialize EnsembleAgent (takes time, run once)
import chromadb
from price_agents.ensemble_agent import EnsembleAgent

print("🧠 Initializing EnsembleAgent (3 models)...")
print("=" * 60)

# Connect to ChromaDB
DB_PATH = "products_vectorstore"
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection('products')

print(f"ChromaDB: {collection.count()} documents")

# Initialize EnsembleAgent
ensemble = EnsembleAgent(collection)
print("\n✅ EnsembleAgent ready!")

INFO:datasets:PyTorch version 2.9.0 available.
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


🧠 Initializing EnsembleAgent (3 models)...


INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


ChromaDB: 800000 documents


INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready



✅ EnsembleAgent ready!


## 🚀 Skip Flow Simulation

When user clicks "Skip, search now":
1. No clarification questions are asked
2. Search directly with the original keyword
3. Pipeline runs as normal

In [4]:
# Cell 4: Skip Flow - User Input
# 📝 CHANGE THIS KEYWORD TO TEST DIFFERENT PRODUCTS

TEST_KEYWORD = "laptop"  # Original keyword (no clarification)
SKIP_CLARIFICATION = True  # Simulate "Skip, search now" button click

print("=" * 60)
print("🚀 SKIP FLOW - Direct Search")
print("=" * 60)
print(f"\n📝 Original keyword: '{TEST_KEYWORD}'")
print(f"⏩ Skip clarification: {SKIP_CLARIFICATION}")

if SKIP_CLARIFICATION:
    SEARCH_QUERY = TEST_KEYWORD  # Use original keyword directly
    print(f"\n✅ Using original keyword for search: '{SEARCH_QUERY}'")
else:
    print("\n⚠️ SKIP_CLARIFICATION is False. Run clarification_agent_test.ipynb instead.")

🚀 SKIP FLOW - Direct Search

📝 Original keyword: 'laptop'
⏩ Skip clarification: True

✅ Using original keyword for search: 'laptop'


In [5]:
# Cell 5: Step 1 - Search URLs with Original Keyword

print("=" * 60)
print(f"🔍 STEP 1: Searching BestBuy for '{SEARCH_QUERY}'")
print("=" * 60)

# Initialize search agent
search_agent = BestBuySearchAgent()

# Search with original keyword (no refinement)
MAX_URLS = 15
urls = search_agent.search(SEARCH_QUERY, max_urls=MAX_URLS)

print(f"\n✅ Found {len(urls)} product URLs:")
for i, url in enumerate(urls[:5], 1):  # Show first 5
    print(f"   {i}. {url[:80]}...")
if len(urls) > 5:
    print(f"   ... and {len(urls) - 5} more")

INFO:root:[BestBuy Search Agent] BestBuy Search Agent is initializing
INFO:root:[BestBuy Search Agent] BestBuy Search Agent is ready
INFO:root:[BestBuy Search Agent] Searching for: laptop


🔍 STEP 1: Searching BestBuy for 'laptop'


INFO:root:[BestBuy Search Agent] Found 15 product URLs



✅ Found 15 product URLs:
   1. https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-s...
   2. https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen-laptop-intel-core-i...
   3. https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen-laptop-intel-core-i...
   4. https://www.bestbuy.com/product/hp-17-3-full-hd-laptop-intel-core-i7-1355U-2023-...
   5. https://www.bestbuy.com/product/lenovo-loq-15-6-full-hd-gaming-laptop-amd-ryzen-...
   ... and 10 more


In [6]:
# Cell 6: Step 2 - Filter Sale URLs

print("=" * 60)
print(f"🏷️ STEP 2: Filtering {len(urls)} URLs for sale items")
print("=" * 60)

sale_urls = filter_sale_urls(urls)

print(f"\n✅ Found {len(sale_urls)} products on SALE:")
for i, url in enumerate(sale_urls, 1):
    print(f"   {i}. {url[:80]}...")

if not sale_urls:
    print("\n⚠️ No sale items found. Try a different keyword.")

🏷️ STEP 2: Filtering 15 URLs for sale items


INFO:price_agents.bestbuy_deals:[1/15] SALE
INFO:price_agents.bestbuy_deals:[2/15] SALE
INFO:price_agents.bestbuy_deals:[3/15] SALE
INFO:price_agents.bestbuy_deals:[4/15] SALE
INFO:price_agents.bestbuy_deals:[5/15] Skip
INFO:price_agents.bestbuy_deals:[6/15] Skip
INFO:price_agents.bestbuy_deals:[7/15] SALE
INFO:price_agents.bestbuy_deals:[8/15] SALE
INFO:price_agents.bestbuy_deals:[9/15] Skip
INFO:price_agents.bestbuy_deals:[10/15] SALE
INFO:price_agents.bestbuy_deals:[11/15] SALE
INFO:price_agents.bestbuy_deals:[12/15] SALE
INFO:price_agents.bestbuy_deals:[13/15] SALE
INFO:price_agents.bestbuy_deals:[14/15] SALE
INFO:price_agents.bestbuy_deals:[15/15] Skip
INFO:price_agents.bestbuy_deals:Filtered 15 URLs → 11 sale URLs



✅ Found 11 products on SALE:
   1. https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled-laptop-copilot-pc-s...
   2. https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen-laptop-intel-core-i...
   3. https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen-laptop-intel-core-i...
   4. https://www.bestbuy.com/product/hp-17-3-full-hd-laptop-intel-core-i7-1355U-2023-...
   5. https://www.bestbuy.com/product/lenovo-yoga-7-2-in-1-copilot-pc-14-2k-oled-touch...
   6. https://www.bestbuy.com/product/hp-omnibook-x-flip-2-in-1-copilot-pc-14-2k-touch...
   7. https://www.bestbuy.com/product/asus-vivobook-14-14-fhd-laptop-intel-core-i3-131...
   8. https://www.bestbuy.com/product/lenovo-ideapad-slim-3-15-6-full-hd-touchscreen-l...
   9. https://www.bestbuy.com/product/lenovo-ideapad-slim-3i-15-6-full-hd-laptop-intel...
   10. https://www.bestbuy.com/product/hp-omnibook-x-flip-2-in-1-14-2k-touch-screen-lap...
   11. https://www.bestbuy.com/product/hp-victus-15-6-144hz-full-hd-g

In [7]:
# Cell 7: Step 3 - Scrape Products with Playwright
# ⚠️ Browser window will open!

print("=" * 60)
print(f"📦 STEP 3: Scraping {len(sale_urls)} products with Playwright")
print("=" * 60)
print("⚠️ Browser window will open...\n")

# Scrape products
scraped_deals = await scrape_bestbuy_products(sale_urls, headless=False)

print(f"\n✅ Scraped {len(scraped_deals)} products:")
for i, deal in enumerate(scraped_deals, 1):
    print(f"\n   [{i}] {deal.title[:60]}...")
    print(f"       💰 ${deal.price}")

📦 STEP 3: Scraping 11 products with Playwright
⚠️ Browser window will open...



INFO:price_agents.bestbuy_deals:[1/11] Scraping: https://www.bestbuy.com/product/asus-zenbook-a14-14-fhd-oled...
INFO:price_agents.bestbuy_deals:  ✓ <ASUS - Zenbook A14 14" FHD+ OLED Laptop - Copilot+... | $739.0>
INFO:price_agents.bestbuy_deals:[2/11] Scraping: https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen...
INFO:price_agents.bestbuy_deals:  ✓ <HP - 15.6" Full HD Touch-Screen Laptop - Intel Cor... | $549.99>
INFO:price_agents.bestbuy_deals:[3/11] Scraping: https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen...
INFO:price_agents.bestbuy_deals:  ✓ <HP - 15.6" Full HD Touch-Screen Laptop - Intel Cor... | $0.0>
INFO:price_agents.bestbuy_deals:[4/11] Scraping: https://www.bestbuy.com/product/hp-17-3-full-hd-laptop-intel...
INFO:price_agents.bestbuy_deals:  ✓ <HP - 17.3" Full HD Laptop - Intel Core i7 1355U 20... | $549.99>
INFO:price_agents.bestbuy_deals:[5/11] Scraping: https://www.bestbuy.com/product/lenovo-yoga-7-2-in-1-copilot...
INFO:price_agents.bestbuy_deals


✅ Scraped 11 products:

   [1] ASUS - Zenbook A14 14" FHD+ OLED Laptop - Copilot+ PC - Snap...
       💰 $739.0

   [2] HP - 15.6" Full HD Touch-Screen Laptop - Intel Core i7 1355U...
       💰 $549.99

   [3] HP - 15.6" Full HD Touch-Screen Laptop - Intel Core i5 1334U...
       💰 $0.0

   [4] HP - 17.3" Full HD Laptop - Intel Core i7 1355U 2023 - 16GB ...
       💰 $549.99

   [5] Lenovo - Yoga 7 2-in-1 - Copilot+ PC - 14" 2K OLED Touchscre...
       💰 $0.0

   [6] HP - OmniBook X Flip 2-in-1 - Copilot+ PC - 14" 2K Touch-Scr...
       💰 $799.99

   [7] ASUS - Vivobook 14 14" FHD Laptop - Intel Core i3-1315U with...
       💰 $249.99

   [8] Lenovo - IdeaPad Slim 3 15.6" Full HD Touchscreen Laptop - A...
       💰 $489.99

   [9] Lenovo - IdeaPad Slim 3i 15.6" Full HD Laptop - Intel Core i...
       💰 $289.99

   [10] HP - OmniBook 5 Flip 2-in-1 14" 2K Touch-Screen Laptop - Int...
       💰 $542.99

   [11] HP - Victus 15.6" 144Hz Full HD Gaming Laptop - AMD Ryzen 7 ...
       💰 $749.0


In [8]:
# Cell 8: Step 4 - Select Top Deals with GPT-5-mini

print("=" * 60)
print(f"🤖 STEP 4: Selecting top 5 deals with GPT-5-mini")
print("=" * 60)

scanner = BestBuyScannerAgent()
deal_selection = scanner.scan(scraped_deals)

if deal_selection and deal_selection.deals:
    print(f"\n✅ Selected {len(deal_selection.deals)} best deals:")
    for i, deal in enumerate(deal_selection.deals, 1):
        print(f"\n   [{i}] {deal.product_description[:80]}...")
        print(f"       💰 ${deal.price}")
else:
    print("\n⚠️ No deals selected. Check scraped_deals.")

INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is initializing
INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is ready
INFO:root:[BestBuy Scanner Agent] Calling gpt-5-mini with 9 deals...


🤖 STEP 4: Selecting top 5 deals with GPT-5-mini


INFO:root:[BestBuy Scanner Agent] Selected 5 deals



✅ Selected 5 best deals:

   [1] The ASUS Zenbook A14 is a thin, lightweight 14-inch laptop built in a Ceraluminu...
       💰 $739.0

   [2] The HP OmniBook X Flip is a 14-inch 2-in-1 convertible with an edge-to-edge 2K m...
       💰 $799.99

   [3] The HP Victus 15.6-inch gaming laptop pairs an AMD Ryzen 7 7445HS processor with...
       💰 $749.0

   [4] The Lenovo IdeaPad Slim 3 is a 15.6-inch Full HD touchscreen laptop built with a...
       💰 $489.99

   [5] The HP OmniBook 5 Flip is a 14-inch 2-in-1 convertible with a 2K touch display (...
       💰 $542.99


In [9]:
# Cell 9: Step 5 - Estimate Prices & Calculate Discounts

print("=" * 60)
print(f"💰 STEP 5: Estimating prices with EnsembleAgent")
print("=" * 60)

opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f"\n[{i}/{len(deal_selection.deals)}] Estimating: {deal.product_description[:50]}...")
    
    # Get price estimate from EnsembleAgent
    estimate = ensemble.price(deal.product_description)
    discount = estimate - deal.price
    
    # Create Opportunity
    opportunity = Opportunity(
        deal=deal,
        estimate=estimate,
        discount=discount
    )
    opportunities.append(opportunity)
    
    print(f"   💵 Sale Price:  ${deal.price:.2f}")
    print(f"   📊 Estimate:    ${estimate:.2f}")
    print(f"   🏷️  Discount:    ${discount:.2f}")

# Sort by discount (highest first)
opportunities.sort(key=lambda x: x.discount, reverse=True)

print(f"\n✅ Estimated {len(opportunities)} opportunities!")

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text


💰 STEP 5: Estimating prices with EnsembleAgent

[1/5] Estimating: The ASUS Zenbook A14 is a thin, lightweight 14-inc...


11:05:35 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
11:05:36 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $700.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $899.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $667.30
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $855.93
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
11:06:25 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


...

   💵 Sale Price:  $489.99
   📊 Estimate:    $683.00
   🏷️  Discount:    $193.01

[5/5] Estimating: The HP OmniBook 5 Flip is a 14-inch 2-in-1 convert...


11:06:35 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $700.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $749.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $784.27
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $747.63


   💵 Sale Price:  $542.99
   📊 Estimate:    $747.63
   🏷️  Discount:    $204.64

✅ Estimated 5 opportunities!


In [10]:
# Cell 10: Final Results - Display Table

print("=" * 80)
print("🏆 FINAL RESULTS - Skip Flow (Sorted by Discount)")
print("=" * 80)
print(f"\n📝 Search keyword: '{SEARCH_QUERY}'")
print(f"⏩ Clarification: SKIPPED")
print(f"📊 Found {len(opportunities)} deals\n")

for i, opp in enumerate(opportunities, 1):
    discount_pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    
    # Status emoji based on discount
    if opp.discount > 200:
        status = "🔥 HOT DEAL!"
    elif opp.discount > 100:
        status = "✅ Good Deal"
    elif opp.discount > 0:
        status = "👍 OK"
    else:
        status = "❌ Overpriced"
    
    print(f"{'─' * 80}")
    print(f"#{i} {status}")
    print(f"   📦 {opp.deal.product_description[:70]}...")
    print(f"   💵 Sale Price:     ${opp.deal.price:.2f}")
    print(f"   📊 Estimated Value: ${opp.estimate:.2f}")
    print(f"   🏷️  Discount:        ${opp.discount:.2f} ({discount_pct:.1f}%)")
    print(f"   🔗 {opp.deal.url}")

print(f"\n{'=' * 80}")
print(f"📊 Summary:")
print(f"   - Total deals: {len(opportunities)}")
print(f"   - Positive discount: {len([o for o in opportunities if o.discount > 0])}")
print(f"   - Great deals (>$100): {len([o for o in opportunities if o.discount > 100])}")
if opportunities:
    print(f"   - Best discount: ${opportunities[0].discount:.2f}")

🏆 FINAL RESULTS - Skip Flow (Sorted by Discount)

📝 Search keyword: 'laptop'
⏩ Clarification: SKIPPED
📊 Found 5 deals

────────────────────────────────────────────────────────────────────────────────
#1 🔥 HOT DEAL!
   📦 The HP OmniBook 5 Flip is a 14-inch 2-in-1 convertible with a 2K touch...
   💵 Sale Price:     $542.99
   📊 Estimated Value: $747.63
   🏷️  Discount:        $204.64 (27.4%)
   🔗 https://www.bestbuy.com/product/hp-omnibook-x-flip-2-in-1-14-2k-touch-screen-laptop-intel-core-ultra-5-8gb-memory-512gb-ssd-glacier-silver/JJGQJRKGTQ
────────────────────────────────────────────────────────────────────────────────
#2 ✅ Good Deal
   📦 The HP Victus 15.6-inch gaming laptop pairs an AMD Ryzen 7 7445HS proc...
   💵 Sale Price:     $749.00
   📊 Estimated Value: $945.91
   🏷️  Discount:        $196.91 (20.8%)
   🔗 https://www.bestbuy.com/product/hp-victus-15-6-144hz-full-hd-gaming-laptop-amd-ryzen-7-7445hs-2023-16gb-memory-nvidia-geforce-rtx-4050-512gb-ssd-mica-silver/JJGH2L8JVV
─────

## 📊 Comparison: Skip Flow vs Clarification Flow

| Aspect | Skip Flow | Clarification Flow |
|--------|-----------|-------------------|
| **User Input** | Just keyword | Keyword + 3 answers |
| **Search Query** | `laptop` | `Acer laptops for students under $800` |
| **Results** | Broad/General | Targeted/Specific |
| **Speed** | Faster | Slightly slower (1 extra LLM call) |
| **Best For** | Users who know what they want | Users who need guidance |

## ✅ Skip Flow Test Complete!

**Next Steps:**
1. ✅ Skip Flow tested
2. 🔲 Integrate both flows into Gradio UI
3. 🔲 Add "Skip, search now" button to clarification UI